# Step 1 — Players Data Collection Setup

## FIFA World Cup 2026 Data Analytics

This notebook is dedicated ONLY to collecting the Players table.

The final dataset will contain:

- PlayerID
- PlayerName
- TeamID
- Position
- JerseyNumber

Rules:

- Use real FIFA World Cup 2026 player data.
- Do not generate fake players.
- Do not invent player information.
- TeamID must come from the existing Teams.csv dataset.
- Do not modify the completed Teams.csv dataset.
- This notebook must remain separate from all other table notebooks.

In [8]:
import requests
import pandas as pd
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Test FIFA Player Data Source

The FIFA World Cup 2026 official squad list contains the player
shirt number and position for the 48 participating teams.

We will first download the official FIFA squad list PDF and verify
that it contains the required player information before extracting
the complete dataset.

In [9]:
import requests
from pathlib import Path

pdf_url = "https://fdp.fifa.org/assetspublic/ce281/pdf/SquadLists-English.pdf"

response = requests.get(pdf_url, timeout=30)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("File size:", len(response.content), "bytes")

Status code: 200
Content type: application/pdf
File size: 784283 bytes


## 3. Download the Official FIFA Squad List

The official FIFA World Cup 2026 squad list PDF will be saved locally.

This file will be used as the source for extracting:
- Player name
- Team
- Position
- Jersey number

In [10]:
from pathlib import Path
import requests

pdf_url = "https://fdp.fifa.org/assetspublic/ce281/pdf/SquadLists-English.pdf"

pdf_path = Path("../Data/FIFA_WorldCup_2026_SquadLists.pdf")

response = requests.get(pdf_url, timeout=30)
response.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully!")
print("Saved to:", pdf_path.resolve())
print("File size:", pdf_path.stat().st_size, "bytes")

PDF downloaded successfully!
Saved to: D:\ai course hw\FIFA_WorldCup_2026_Project\Data\FIFA_WorldCup_2026_SquadLists.pdf
File size: 784283 bytes


## 4. Inspect the Official FIFA Squad List

We will extract the text from the official FIFA squad list PDF.

The extracted text will be inspected to understand its structure
before creating the Players dataset.

In [11]:
import PyPDF2

with open(pdf_path, "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)

    print("Number of pages:", len(pdf_reader.pages))

    text = pdf_reader.pages[0].extract_text()

    print("\nFirst page preview:\n")
    print(text[:3000])

Number of pages: 48

First page preview:

Algeria (AL G)
#POS PLAYER NAME FIRST NAME(S) LAST NAME(S) NAME ON SHIR T DOB CLUB HEIGHT (CM) CAPS GOALS
GKMASTIL Melvin Melvin Feycal MASTIL MASTIL 19/02/2000 FC Stade Ny onnais (SUI) 194 2 0
DFMANDI Aissa Aissa MANDI MANDI 22/10/1991 Lille OSC (FRA) 184 123 8
DFABADA Achref Achref ABADA ABADA 15/06/1999 USM Alger (ALG) 185 10 1
DFTOUGAI Mohamed Amine Mohamed Amine TOUGAI TOUGAI 22/01/2000 Espérance De T unisie (TUN) 186 30 2
DFBELAID Zineddine Zineddine BELAÏD BELAID 20/03/1999 JS Kabylie (ALG) 186 20 1
MFZERROUKI Ramiz Ramiz Larbi ZERROUKI ZERROUKI 26/05/1998 FC Twente (NED) 183 56 3
FWMAHREZ Riy ad Riyad Karim MAHREZ MAHREZ 21/02/1991 Al Ahli FC (KSA) 179 120 40
MFAOUAR Houssem Houssem-E ddine Chaabane AOUAR AOUAR 30/06/1998 Al Ittihad (KSA) 175 26 6
FWGOUIRI Amine Amine Ferid GHOUIRI GOUIRI 16/02/2000 Olympique Marseille (FRA) 180 27 11
MFCHAIBI Fares Fares CHAIBI CHAIBI 28/11/2002 Eintracht Frankfurt (GER) 183 35 3
FWHADJ MOUSSA Anis Ani

## 5. Extract All Squad Text

We will extract the text from all 48 pages of the official FIFA
World Cup 2026 Squad List.

The complete extracted text will be stored in memory so that we can
process the teams and players in the next steps.

In [12]:
with open(pdf_path, "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)

    all_pages_text = []

    for page in pdf_reader.pages:
        page_text = page.extract_text()
        all_pages_text.append(page_text)

full_text = "\n".join(all_pages_text)

print("Pages extracted:", len(all_pages_text))
print("Total characters extracted:", len(full_text))
print("\nFirst 1000 characters:\n")
print(full_text[:1000])

Pages extracted: 48
Total characters extracted: 133258

First 1000 characters:

Algeria (AL G)
#POS PLAYER NAME FIRST NAME(S) LAST NAME(S) NAME ON SHIR T DOB CLUB HEIGHT (CM) CAPS GOALS
GKMASTIL Melvin Melvin Feycal MASTIL MASTIL 19/02/2000 FC Stade Ny onnais (SUI) 194 2 0
DFMANDI Aissa Aissa MANDI MANDI 22/10/1991 Lille OSC (FRA) 184 123 8
DFABADA Achref Achref ABADA ABADA 15/06/1999 USM Alger (ALG) 185 10 1
DFTOUGAI Mohamed Amine Mohamed Amine TOUGAI TOUGAI 22/01/2000 Espérance De T unisie (TUN) 186 30 2
DFBELAID Zineddine Zineddine BELAÏD BELAID 20/03/1999 JS Kabylie (ALG) 186 20 1
MFZERROUKI Ramiz Ramiz Larbi ZERROUKI ZERROUKI 26/05/1998 FC Twente (NED) 183 56 3
FWMAHREZ Riy ad Riyad Karim MAHREZ MAHREZ 21/02/1991 Al Ahli FC (KSA) 179 120 40
MFAOUAR Houssem Houssem-E ddine Chaabane AOUAR AOUAR 30/06/1998 Al Ittihad (KSA) 175 26 6
FWGOUIRI Amine Amine Ferid GHOUIRI GOUIRI 16/02/2000 Olympique Marseille (FRA) 180 27 11
MFCHAIBI Fares Fares CHAIBI CHAIBI 28/11/2002 Eintracht Frankfurt

## 6. Inspect Squad Page Structure

The PDF text extraction combines some columns together.

We will inspect the beginning of several squad pages to identify a
reliable pattern for extracting:

- Team name
- Jersey number
- Position
- Player name

In [13]:
for i in [0, 1, 2]:
    print("=" * 80)
    print(f"PAGE {i + 1}")
    print("=" * 80)
    print(all_pages_text[i][:2500])
    print()

PAGE 1
Algeria (AL G)
#POS PLAYER NAME FIRST NAME(S) LAST NAME(S) NAME ON SHIR T DOB CLUB HEIGHT (CM) CAPS GOALS
GKMASTIL Melvin Melvin Feycal MASTIL MASTIL 19/02/2000 FC Stade Ny onnais (SUI) 194 2 0
DFMANDI Aissa Aissa MANDI MANDI 22/10/1991 Lille OSC (FRA) 184 123 8
DFABADA Achref Achref ABADA ABADA 15/06/1999 USM Alger (ALG) 185 10 1
DFTOUGAI Mohamed Amine Mohamed Amine TOUGAI TOUGAI 22/01/2000 Espérance De T unisie (TUN) 186 30 2
DFBELAID Zineddine Zineddine BELAÏD BELAID 20/03/1999 JS Kabylie (ALG) 186 20 1
MFZERROUKI Ramiz Ramiz Larbi ZERROUKI ZERROUKI 26/05/1998 FC Twente (NED) 183 56 3
FWMAHREZ Riy ad Riyad Karim MAHREZ MAHREZ 21/02/1991 Al Ahli FC (KSA) 179 120 40
MFAOUAR Houssem Houssem-E ddine Chaabane AOUAR AOUAR 30/06/1998 Al Ittihad (KSA) 175 26 6
FWGOUIRI Amine Amine Ferid GHOUIRI GOUIRI 16/02/2000 Olympique Marseille (FRA) 180 27 11
MFCHAIBI Fares Fares CHAIBI CHAIBI 28/11/2002 Eintracht Frankfurt (GER) 183 35 3
FWHADJ MOUSSA Anis Anis HADJ MOUSSA HADJ MOUSSA 11/02/200

## 7. Test Player Row Extraction

The FIFA PDF combines the position and player number during text
extraction.

We will use the repeated position markers (GK, DF, MF, FW) to identify
the beginning of each player record.

This step only tests the extraction on the first team before applying
the parser to all 48 teams.

In [14]:
import re

page_text = all_pages_text[0]

# Find all player records beginning with GK, DF, MF, or FW
player_starts = list(re.finditer(r'\b(?:GK|DF|MF|FW)', page_text))

print("Possible player records found:", len(player_starts))

print("\nFirst 10 detected records:\n")

for i, match in enumerate(player_starts[:10], start=1):
    start = match.start()
    end = player_starts[i].start() if i < len(player_starts) else start + 150

    record = page_text[start:end].strip()

    print(f"{i}. {record[:200]}")

Possible player records found: 30

First 10 detected records:

1. GKMASTIL Melvin Melvin Feycal MASTIL MASTIL 19/02/2000 FC Stade Ny onnais (SUI) 194 2 0
2. DFMANDI Aissa Aissa MANDI MANDI 22/10/1991 Lille OSC (FRA) 184 123 8
3. DFABADA Achref Achref ABADA ABADA 15/06/1999 USM Alger (ALG) 185 10 1
4. DFTOUGAI Mohamed Amine Mohamed Amine TOUGAI TOUGAI 22/01/2000 Espérance De T unisie (TUN) 186 30 2
5. DFBELAID Zineddine Zineddine BELAÏD BELAID 20/03/1999 JS Kabylie (ALG) 186 20 1
6. MFZERROUKI Ramiz Ramiz Larbi ZERROUKI ZERROUKI 26/05/1998 FC Twente (NED) 183 56 3
7. FWMAHREZ Riy ad Riyad Karim MAHREZ MAHREZ 21/02/1991 Al Ahli FC (KSA) 179 120 40
8. MFAOUAR Houssem Houssem-E ddine Chaabane AOUAR AOUAR 30/06/1998 Al Ittihad (KSA) 175 26 6
9. FWGOUIRI Amine Amine Ferid GHOUIRI GOUIRI 16/02/2000 Olympique Marseille (FRA) 180 27 11
10. MFCHAIBI Fares Fares CHAIBI CHAIBI 28/11/2002 Eintracht Frankfurt (GER) 183 35 3


## 8. Inspect Jersey Number Extraction

The extracted PDF text does not preserve the jersey-number column
cleanly.

We will inspect the complete player-row structure to determine a
reliable way to extract the jersey number without guessing or
generating data.

In [15]:
for i, match in enumerate(player_starts[:25], start=1):
    start = match.start()
    end = player_starts[i].start() if i < len(player_starts) else start + 200

    record = page_text[start:end].strip()

    print(f"{i}. {record}")

1. GKMASTIL Melvin Melvin Feycal MASTIL MASTIL 19/02/2000 FC Stade Ny onnais (SUI) 194 2 0
2. DFMANDI Aissa Aissa MANDI MANDI 22/10/1991 Lille OSC (FRA) 184 123 8
3. DFABADA Achref Achref ABADA ABADA 15/06/1999 USM Alger (ALG) 185 10 1
4. DFTOUGAI Mohamed Amine Mohamed Amine TOUGAI TOUGAI 22/01/2000 Espérance De T unisie (TUN) 186 30 2
5. DFBELAID Zineddine Zineddine BELAÏD BELAID 20/03/1999 JS Kabylie (ALG) 186 20 1
6. MFZERROUKI Ramiz Ramiz Larbi ZERROUKI ZERROUKI 26/05/1998 FC Twente (NED) 183 56 3
7. FWMAHREZ Riy ad Riyad Karim MAHREZ MAHREZ 21/02/1991 Al Ahli FC (KSA) 179 120 40
8. MFAOUAR Houssem Houssem-E ddine Chaabane AOUAR AOUAR 30/06/1998 Al Ittihad (KSA) 175 26 6
9. FWGOUIRI Amine Amine Ferid GHOUIRI GOUIRI 16/02/2000 Olympique Marseille (FRA) 180 27 11
10. MFCHAIBI Fares Fares CHAIBI CHAIBI 28/11/2002 Eintracht Frankfurt (GER) 183 35 3
11. FWHADJ MOUSSA Anis Anis HADJ MOUSSA HADJ MOUSSA 11/02/2002 Feyenoord Rotterdam (NED) 176 18 2
12. FWBENBOU ALI Nadhir Ahmed Nadhir BENB

## 9. Extract Player Data Using PDF Coordinates

The standard text extraction does not preserve the jersey-number
column.

We will use pdfplumber to extract words together with their positions
on the page.

This allows us to identify the jersey-number column based on its
location instead of guessing from the text.

In [16]:
import pdfplumber

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()

print("Number of extracted words:", len(words))

print("\nFirst 50 words with coordinates:\n")

for word in words[:50]:
    print(
        f"Text: {word['text']:<20} "
        f"x0: {word['x0']:.1f} "
        f"top: {word['top']:.1f}"
    )

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Number of extracted words: 469

First 50 words with coordinates:

Text: SQUAD                x0: 208.4 top: 4.1
Text: LIST                 x0: 219.2 top: 4.1
Text: FIFA                 x0: 11.1 top: 10.8
Text: World                x0: 31.4 top: 10.8
Text: Cup                  x0: 57.8 top: 10.8
Text: 2026™                x0: 76.4 top: 10.8
Text: 11                   x0: 11.1 top: 22.7
Text: June                 x0: 16.7 top: 22.7
Text: 2026                 x0: 26.8 top: 22.7
Text: –                    x0: 37.1 top: 22.7
Text: 19                   x0: 40.8 top: 22.7
Text: July                 x0: 46.4 top: 22.7
Text: 2026                 x0: 55.0 top: 22.7
Text: Algeria              x0: 204.6 top: 36.1
Text: (ALG)                x0: 218.8 top: 36.1
Text: #                    x0: 8.4 top: 44.3
Text: POS                  x0: 16.8 top: 44.3
Text: PLAYER               x0: 49.4 top: 44.3
Text: NAME                 x0: 61.4 top: 44.3
Text: FIRST                x0: 117.6 top: 44.3
Text: NAME(S

## 10. Inspect the First Player Row

We will inspect the first player's words and their horizontal
positions.

This will allow us to identify the exact column used for the
jersey number.

In [17]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()

# The first player row starts around this vertical position.
first_player_words = [
    word for word in words
    if 53 <= word["top"] <= 58
]

print("Words found in first player row:\n")

for word in first_player_words:
    print(
        f"Text: {word['text']:<20} "
        f"x0: {word['x0']:.1f} "
        f"top: {word['top']:.1f}"
    )

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Words found in first player row:

Text: 2                    x0: 8.4 top: 54.6
Text: DF                   x0: 18.0 top: 54.6
Text: MANDI                x0: 28.3 top: 54.6
Text: Aissa                x0: 38.9 top: 54.6
Text: Aissa                x0: 94.8 top: 54.6
Text: MANDI                x0: 166.0 top: 54.6
Text: MANDI                x0: 208.2 top: 54.6
Text: 22/10/1991           x0: 257.7 top: 54.6
Text: Lille                x0: 283.5 top: 54.6
Text: OSC                  x0: 289.8 top: 54.6
Text: (FRA)                x0: 296.6 top: 54.6
Text: 184                  x0: 369.3 top: 54.6
Text: 123                  x0: 395.8 top: 54.6
Text: 8                    x0: 416.9 top: 54.6


## 11. Extract Player Rows, Names, Position and Team (rewritten)

Earlier attempts mixed FIRST NAME(S) / LAST NAME(S) / NAME ON SHIRT into
the result. This section instead:

- Uses **only** the official `PLAYER NAME` column (ignores FIRST NAME(S),
  LAST NAME(S) and NAME ON SHIRT completely).
- Splits each row into columns using the horizontal gaps between words
  (robust to the exact template of the PDF).
- Reorders the `PLAYER NAME` field into `Givenname Surname` order.

Why the reordering needs a rule instead of a blind swap: the PDF writes
`PLAYER NAME` as `SURNAME Givenname` most of the time (e.g. `MASTIL Melvin`,
`MBAPPE Kylian`), but for players commonly known by two forenames it is
**already** `Givenname Givenname`, written fully in caps (e.g.
`CRISTIANO RONALDO`, `GONCALO RAMOS`, `NEYMAR JR`). Blindly reversing every
name would turn `CRISTIANO RONALDO` into `Ronaldo Cristiano`, which is wrong.

The rule used below: scan the words in `PLAYER NAME` for the first one that
looks like a normal Title-Case given name (e.g. `Scott`, `Anis`). Everything
before it is the surname and gets moved to the end. If no word looks like a
normal given name, the field is already in the correct order and is only
re-cased.

In [18]:
import re

def split_name_groups(row_words):
    """Split the words between POS and DOB into 4 column groups:
    [PLAYER NAME] [FIRST NAME(S)] [LAST NAME(S)] [NAME ON SHIRT]
    using the biggest horizontal gaps as the column boundaries."""
    dob_idx = None
    for i, w in enumerate(row_words):
        if re.match(r'^\d{2}/\d{2}/\d{4}$', w["text"]):
            dob_idx = i
            break
    name_words = row_words[:dob_idx]

    gaps = []
    for i in range(1, len(name_words)):
        gap = name_words[i]["x0"] - name_words[i - 1]["x1"]
        gaps.append((gap, i))
    gaps.sort(reverse=True)
    split_points = sorted(g[1] for g in gaps[:3])

    groups, prev = [], 0
    for sp in split_points:
        groups.append(name_words[prev:sp])
        prev = sp
    groups.append(name_words[prev:])
    return [" ".join(w["text"] for w in g) for g in groups]


def _is_titlecase_word(token):
    """True for a normal given-name word like 'Scott', N'Golo, 'Jean-Baptiste'.
    False for surname-style words like 'MASTIL', 'McTOMINAY', 'AIT-NOURI'."""
    for part in re.split(r"[-']", token):
        if len(part) <= 1:
            continue
        if not (part[0].isupper() and part[1:].islower()):
            return False
    return True


def natural_order_name(player_name_field):
    """Reorder the PLAYER NAME field into 'Givenname Surname' order.
    See markdown cell above for the reasoning."""
    tokens = player_name_field.split()
    if len(tokens) <= 1:
        return player_name_field.title()

    split_at = next((i for i, t in enumerate(tokens) if _is_titlecase_word(t)), None)

    if split_at is None or split_at == 0:
        return player_name_field.title()

    surname = " ".join(tokens[:split_at]).title()
    given = " ".join(tokens[split_at:])
    return f"{given} {surname}"


all_player_data = []

with pdfplumber.open(pdf_path) as pdf:

    for page_number, page in enumerate(pdf.pages, start=1):

        words = page.extract_words()

        header_words = [w["text"] for w in words if 30 <= w["top"] <= 42]
        team_header = " ".join(header_words)
        team_name = re.sub(r"\s*\([A-Z]{3}\)$", "", team_header).strip()

        for w in words:
            if not (w["text"].isdigit() and w["x0"] < 15):
                continue

            row_top = w["top"]
            row = sorted(
                (x for x in words if abs(x["top"] - row_top) < 1),
                key=lambda x: x["x0"],
            )
            if len(row) < 3 or row[1]["text"] not in ("GK", "DF", "MF", "FW"):
                continue

            jersey = int(row[0]["text"])
            position = row[1]["text"]

            player_name_raw, _first, _last, _shirt = split_name_groups(row[2:])
            # The PDF font drops the "fi" ligature glyph as a null character
            # in a handful of names (e.g. "Ra\x00k" -> should be "Rafik").
            player_name_raw = player_name_raw.replace("\x00", "fi")

            all_player_data.append({
                "Page": page_number,
                "TeamName": team_name,
                "JerseyNumber": jersey,
                "Position": position,
                "PlayerName": natural_order_name(player_name_raw),
            })

players_df = pd.DataFrame(all_player_data)

print("Total players extracted:", len(players_df))
print("Teams found:", players_df["TeamName"].nunique())
print("Missing PlayerName values:", players_df["PlayerName"].isna().sum())

print("\nFirst 20 players:\n")
print(players_df.head(20).to_string(index=False))


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Total players extracted: 1248
Teams found: 48
Missing PlayerName values: 0

First 20 players:

 Page TeamName  JerseyNumber Position           PlayerName
    1  Algeria             1       GK        Melvin Mastil
    1  Algeria             2       DF          Aissa Mandi
    1  Algeria             3       DF         Achref Abada
    1  Algeria             4       DF Mohamed Amine Tougai
    1  Algeria             5       DF     Zineddine Belaid
    1  Algeria             6       MF       Ramiz Zerrouki
    1  Algeria             7       FW         Riyad Mahrez
    1  Algeria             8       MF        Houssem Aouar
    1  Algeria             9       FW         Amine Gouiri
    1  Algeria            10       MF         Fares Chaibi
    1  Algeria            11       FW     Anis Hadj Moussa
    1  Algeria            12       FW     Nadhir Benbouali
    1  Algeria            13       DF        Jaouen Hadjam
    1  Algeria            14       MF      Hicham Boudaoui
    1  Algeria      

## 12. Spot-check tricky names

A quick check that both naming conventions in the PDF are handled
correctly (surname-first names get swapped, already-natural-order
names like Cristiano Ronaldo are left alone, and the dropped-ligature
names like Rafik/Alfie/Sofiane are repaired).

In [19]:
for needle in ["Mbappe", "Ronaldo", "Neymar", "Ramos", "Mctominay",
               "Allister", "Belghali", "Jones", "Rahimi"]:
    hits = players_df.loc[players_df["PlayerName"].str.contains(needle), "PlayerName"]
    print(hits.to_string(index=False))


Kylian Mbappe
Cristiano Ronaldo
Neymar Jr
Jiovany Ramos
Goncalo Ramos
Scott Mctominay
Alexis Mac Allister
Rafik Belghali
Alfie Jones
Soufiane Rahimi


## 13. Map Players to TeamID

The SQL Players table uses TeamID as a foreign key. We load the
existing Teams.csv dataset and map each player's team name to the
corresponding TeamID (case/whitespace-insensitive match).

In [21]:
from pathlib import Path
import pandas as pd

# Data folder
data_folder = Path("../Data")

# Load Teams.csv
teams_df = pd.read_csv(data_folder / "Teams.csv")

# Normalize team names for matching
def normalize_team_name(name):
    return " ".join(str(name).lower().split())

# Create TeamName -> TeamID mapping
normalized_team_id_map = {
    normalize_team_name(team_name): team_id
    for team_name, team_id in zip(
        teams_df["TeamName"],
        teams_df["TeamID"]
    )
}

# Assign TeamID to each player
players_df["TeamID"] = players_df["TeamName"].apply(
    lambda x: normalized_team_id_map.get(normalize_team_name(x))
)

# Save the final Players.csv
players_csv_path = data_folder / "Players.csv"
players_df.to_csv(players_csv_path, index=False)

# Check results
print("Total players:", len(players_df))
print("Missing TeamIDs:", players_df["TeamID"].isna().sum())
print("Players.csv saved to:", players_csv_path.resolve())
print()
print(players_df.head(10).to_string(index=False))

Total players: 1248
Missing TeamIDs: 0
Players.csv saved to: D:\ai course hw\FIFA_WorldCup_2026_Project\Data\Players.csv

 Page TeamName  JerseyNumber Position           PlayerName  TeamID
    1  Algeria             1       GK        Melvin Mastil      25
    1  Algeria             2       DF          Aissa Mandi      25
    1  Algeria             3       DF         Achref Abada      25
    1  Algeria             4       DF Mohamed Amine Tougai      25
    1  Algeria             5       DF     Zineddine Belaid      25
    1  Algeria             6       MF       Ramiz Zerrouki      25
    1  Algeria             7       FW         Riyad Mahrez      25
    1  Algeria             8       MF        Houssem Aouar      25
    1  Algeria             9       FW         Amine Gouiri      25
    1  Algeria            10       MF         Fares Chaibi      25


## 14. Assign PlayerID and Build the Final Players Dataset

The final SQL-ready dataset contains exactly:

- PlayerID
- PlayerName
- TeamID
- Position
- JerseyNumber

In [22]:
players_df.insert(0, "PlayerID", range(1, len(players_df) + 1))

players_final_df = players_df[
    ["PlayerID", "PlayerName", "TeamID", "Position", "JerseyNumber"]
].copy()

print("Rows:", len(players_final_df))
print(players_final_df.head(20).to_string(index=False))


Rows: 1248
 PlayerID           PlayerName  TeamID Position  JerseyNumber
        1        Melvin Mastil      25       GK             1
        2          Aissa Mandi      25       DF             2
        3         Achref Abada      25       DF             3
        4 Mohamed Amine Tougai      25       DF             4
        5     Zineddine Belaid      25       DF             5
        6       Ramiz Zerrouki      25       MF             6
        7         Riyad Mahrez      25       FW             7
        8        Houssem Aouar      25       MF             8
        9         Amine Gouiri      25       FW             9
       10         Fares Chaibi      25       MF            10
       11     Anis Hadj Moussa      25       FW            11
       12     Nadhir Benbouali      25       FW            12
       13        Jaouen Hadjam      25       DF            13
       14      Hicham Boudaoui      25       MF            14
       15      Rayan Ait-Nouri      25       DF            

## 15. Validate and Export Players.csv

In [23]:
valid_positions = {"GK", "DF", "MF", "FW"}

print("Total players:", len(players_final_df))
print("Missing values:", players_final_df.isna().sum().sum())
print("Invalid positions:", (~players_final_df["Position"].isin(valid_positions)).sum())
print("Duplicate TeamID + JerseyNumber:",
      players_final_df.duplicated(subset=["TeamID", "JerseyNumber"]).sum())
print("PlayerName max length:", players_final_df["PlayerName"].str.len().max())

players_csv_path = data_folder / "Players.csv"
players_final_df.to_csv(players_csv_path, index=False, encoding="utf-8-sig")
print("\nSaved to:", players_csv_path)


Total players: 1248
Missing values: 0
Invalid positions: 0
Duplicate TeamID + JerseyNumber: 0
PlayerName max length: 24

Saved to: ..\Data\Players.csv
